In [3]:
import sqlite3

# Inicializar la base de datos central del proyecto
conn = sqlite3.connect('bigdata_benchmark.db')
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS lambda_batch_views (
    sensor_id TEXT PRIMARY KEY,
    total_lecturas INTEGER,
    temp_promedio REAL,
    humedad_promedio REAL,
    fecha_procesamiento TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS kappa_stream_views (
    sensor_id TEXT PRIMARY KEY,
    ultima_temperatura REAL,
    ultimo_timestamp TEXT
)
''')

conn.commit()
conn.close()
print("✅ Base de datos 'bigdata_benchmark.db' lista en tu directorio de trabajo.")

✅ Base de datos 'bigdata_benchmark.db' lista en tu directorio de trabajo.


In [1]:
import sqlite3
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg

print("=== INICIANDO BENCHMARK LIGERO: LAMBDA BATCH ===")

# Forzar cierre previo
try:
    spark.stop()
except:
    pass

# Iniciar Spark optimizado para equipos de gama de entrada
spark = SparkSession.builder \
    .appName("Benchmark-Lambda-Batch-Light") \
    .master("local[*]") \
    .config("spark.ui.port", "4042") \
    .config("spark.driver.memory", "1g") \
    .getOrCreate()

# Reducimos a 200,000 registros para evitar cuellos de botella en el i3
print("⚡ Generando y procesando datos...")
df_batch = spark.range(0, 200000) \
    .withColumn("sensor_id", (col("id") % 100).cast("string")) \
    .withColumn("temperatura", (col("id") % 50) + 15.5) \
    .withColumn("humedad", (col("id") % 40) + 30.0)

df_metrics = df_batch.groupBy("sensor_id").agg(
    count("id").alias("total_lecturas"),
    avg("temperatura").alias("temp_promedio"),
    avg("humedad").alias("humedad_promedio")
)

print("⚡ Convirtiendo a Pandas...")
df_pandas = df_metrics.toPandas()

print("💾 Guardando en SQLite...")
conn = sqlite3.connect('bigdata_benchmark.db')
df_pandas.to_sql('lambda_batch_views', conn, if_exists='replace', index=False)
conn.close()

print("✅ ¡Proceso finalizado al instante y guardado con éxito!")

=== INICIANDO BENCHMARK LIGERO: LAMBDA BATCH ===


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/18 00:33:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


⚡ Generando y procesando datos...
⚡ Convirtiendo a Pandas...


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
                                                                                

💾 Guardando en SQLite...
✅ ¡Proceso finalizado al instante y guardado con éxito!


In [7]:
from flask import Flask, jsonify
import sqlite3

app = Flask(__name__)

def query_db(query, args=(), one=False):
    conn = sqlite3.connect('bigdata_benchmark.db')
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()
    cur.execute(query, args)
    rv = cur.fetchall()
    conn.close()
    return (rv[0] if rv else None) if one else rv

@app.route('/api/lambda/metrics', methods=['GET'])
def get_lambda():
    rows = query_db("SELECT * FROM lambda_batch_views LIMIT 5")
    return jsonify([dict(ix) for ix in rows])

@app.route('/api/kappa/metrics', methods=['GET'])
def get_kappa():
    rows = query_db("SELECT * FROM kappa_stream_views LIMIT 5")
    return jsonify([dict(ix) for ix in rows])

if __name__ == '__main__':
    print("🚀 Servidor API simulado listo.")
    # app.run(port=5000)

🚀 Servidor API simulado listo.


In [3]:
!pip install flask

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 1.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 1.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.7/228.7 kB 129.5 kB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [1]:
from flask import Flask, jsonify
import sqlite3

app = Flask(__name__)

def query_db(query, args=(), one=False):
    conn = sqlite3.connect('bigdata_benchmark.db')
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()
    cur.execute(query, args)
    rv = cur.fetchall()
    conn.close()
    return (rv[0] if rv else None) if one else rv

@app.route('/api/lambda/metrics', methods=['GET'])
def get_lambda():
    rows = query_db("SELECT * FROM lambda_batch_views LIMIT 5")
    return jsonify([dict(ix) for ix in rows])

@app.route('/api/kappa/metrics', methods=['GET'])
def get_kappa():
    rows = query_db("SELECT * FROM kappa_stream_views LIMIT 5")
    return jsonify([dict(ix) for ix in rows])

if __name__ == '__main__':
    print("🚀 Servidor API simulado listo.")
    app.run(host='0.0.0.0', port=5000, debug=True)

🚀 Servidor API simulado listo.
 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.18.0.2:5000
Press CTRL+C to quit
 * Restarting with stat
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/local/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/usr/local/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/local/lib/python3.10/site-packages/traitlets/config/application.py", line 1079, in launch_instance
    app.initialize(argv)
  File "/usr/local/lib/python3.10/site-package

SystemExit: 1

/usr/local/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [2]:
import threading
from flask import Flask, jsonify
import sqlite3

app = Flask(__name__)

def query_db(query, args=(), one=False):
    conn = sqlite3.connect('bigdata_benchmark.db')
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()
    cur.execute(query, args)
    rv = cur.fetchall()
    conn.close()
    return (rv[0] if rv else None) if one else rv

@app.route('/api/lambda/metrics', methods=['GET'])
def get_lambda():
    rows = query_db("SELECT * FROM lambda_batch_views LIMIT 5")
    return jsonify([dict(ix) for ix in rows])

@app.route('/api/kappa/metrics', methods=['GET'])
def get_kappa():
    rows = query_db("SELECT * FROM kappa_stream_views LIMIT 5")
    return jsonify([dict(ix) for ix in rows])

def run_server():
    # use_reloader=False evita conflictos de recarga en Jupyter
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

# Lanzar Flask en segundo plano para no bloquear el kernel
server_thread = threading.Thread(target=run_server)
server_thread.daemon = True
server_thread.start()

print("🚀 ¡Servidor Flask corriendo de forma estable en segundo plano (Puerto 5000)!")

🚀 ¡Servidor Flask corriendo de forma estable en segundo plano (Puerto 5000)!
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.18.0.2:5000
Press CTRL+C to quit
172.18.0.1 - - [19/Oct/2026 03:17:00] "GET /api/kappa/metrics HTTP/1.1" 200 -
172.18.0.1 - - [19/Oct/2026 03:17:00] "GET /api/kappa/metrics HTTP/1.1" 200 -
172.18.0.1 - - [19/Oct/2026 03:17:00] "GET /api/kappa/metrics HTTP/1.1" 200 -
172.18.0.1 - - [19/Oct/2026 03:17:00] "GET /api/kappa/metrics HTTP/1.1" 200 -
172.18.0.1 - - [19/Oct/2026 03:17:00] "GET /api/kappa/metrics HTTP/1.1" 200 -
172.18.0.1 - - [19/Oct/2026 03:17:00] "GET /api/kappa/metrics HTTP/1.1" 200 -
172.18.0.1 - - [19/Oct/2026 03:17:00] "GET /api/kappa/metrics HTTP/1.1" 200 -
172.18.0.1 - - [19/Oct/2026 03:17:00] "GET /api/kappa/metrics HTTP/1.1" 200 -
172.18.0.1 - - [19/Oct/2026 03:17:00] "GET /api/kappa/metrics HTTP/1.1" 200 -
172.18.0.1 - - [19/Oct/2026 03:17:00] "GET /api/kappa/metrics HTTP/1.1" 200 -
172.18.0.1 - - [19/Oct/2026 03:17:00] "GET /api/kappa/metrics HTTP/1.1" 200 -
172.18.0.1

In [1]:
!pip install fastapi uvicorn

import asyncio
import sqlite3
import threading
from fastapi import FastAPI
import uvicorn

app = FastAPI(title="Benchmark API - FastAPI (ASGI)")


def query_db(query: str):
    conn = sqlite3.connect("bigdata_benchmark.db")
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()
    cur.execute(query)
    rows = cur.fetchall()
    conn.close()
    return [dict(ix) for ix in rows]


@app.get("/api/lambda/metrics")
async def get_lambda_fastapi():
    # Ejecución no bloqueante para alta concurrencia
    return await asyncio.to_thread(
        query_db, "SELECT * FROM lambda_batch_views LIMIT 5"
    )


@app.get("/api/kappa/metrics")
async def get_kappa_fastapi():
    return await asyncio.to_thread(
        query_db, "SELECT * FROM kappa_stream_views LIMIT 5"
    )


def run_fastapi():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")


# Iniciar FastAPI en segundo plano
fastapi_thread = threading.Thread(target=run_fastapi, daemon=True)
fastapi_thread.start()

print(
    "🚀 ¡Servidor FastAPI (ASGI Asíncrono) corriendo en http://localhost:8000!"
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.7/144.7 kB 945.9 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.0/79.0 kB 1.9 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.1/488.1 kB 3.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 11.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 6.6 MB/s eta 0:00:0000:0100:01m

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
🚀 ¡Servidor FastAPI (ASGI Asíncrono) corriendo en http://localhost:8000!
